# Controlled changes with the idea as a research plan

The judge-prompt ablations of [two_track_figure.ipynb](two_track_figure.ipynb), rerun
with every idea rewritten as a research plan instead of an abstract, and measured
against the **same reference as that figure** — the regular prompt on abstracts. The
strip therefore prints the same numbers in both figures, and a row here is what you
get by switching to plan format *and* applying the change.

The first row is the format change on its own, so the two effects can be told apart:
same row in the two figures differs by roughly what that row's cell here differs
from the first row. P3–P5 were only run pairwise, so there is no pointwise figure.

Reads the chart data of the `merge_plan_format_{hvh,vanilla}.yaml` merges.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src").is_dir() and (p / "output").is_dir())
sys.path.insert(0, str(ROOT / "src"))

from novelty_eval.figures.common import FONT_STACK, PDF_RCPARAMS, TwoTrackFigure

plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": FONT_STACK,
                     **PDF_RCPARAMS})
print(ROOT)

In [ ]:
IDEA = "Idea as research plan"
ROWS = [("convert_to_plan_form_{setup}_{track}", IDEA),
        ("ai_researcher_base", 'P3 · "Which was\njudged novel?"'),
        ("ai_researcher_no_review", 'P4 · "Which is novel?"'),
        ("ai_researcher_comparative", 'P5 · "Which was judged\n more novel?"')]
MODELS = ["claude-sonnet-4-5", "claude-opus-4-5", "claude-opus-4-6",
          "gpt-5.1", "gpt-5.2", "gpt-5.4"]
METRICS = {"pairwise": "pairwise_accuracy_strict"}
GROUPS = [("Judge prompt", ("ai_researcher_base", "ai_researcher_no_review",
                           "ai_researcher_comparative"))]
# track title -> {track} token in a panel name
TRACKS = {"Human-only": "human", "Human + generated": "vanilla"}


def build(merges):
    return TwoTrackFigure(root=ROOT, figure="plan-format",
                          tracks=[(t, merges[t], TRACKS[t]) for t in TRACKS],
                          rows=ROWS, models=MODELS, metrics=METRICS, row_groups=GROUPS,
                          row_axis_label="Change (idea as research plan)",
                          baseline_label="Reference",
                          sig_note="* p < 0.05 (two-sided paired bootstrap)")


# The main figure's merges, where P3-P5 run on abstracts. Only read, for the
# comparison below; its first row is the same panel as ours.
MAIN = build({"Human-only": "merge_hvh_all_metrics",
              "Human + generated": "merge_vanilla_all_metrics"})
# The plan-format merges carry `baseline: <track>_current`, so every panel is
# measured against the same abstract-format reference the main figure uses.
PLAN = build({"Human-only": "merge_plan_format_hvh",
              "Human + generated": "merge_plan_format_vanilla"})

In [ ]:
KEY = ["track", "ablation", "judge"]
TAB = {k: f.table("pairwise").set_index(KEY) for k, f in [("abs", MAIN), ("plan", PLAN)]}
CMP = (pd.DataFrame({"acc_abs": TAB["abs"].ablated, "acc_plan": TAB["plan"].ablated,
                     "d_abs": TAB["abs"].delta, "d_plan": TAB["plan"].delta,
                     "sig_abs": TAB["abs"].significant, "sig_plan": TAB["plan"].significant})
       .drop(IDEA, level="ablation").dropna(subset=["d_abs", "d_plan"]).reset_index())
# If the two changes added up, d_plan would be d_abs plus the format change's own
# delta for that judge. The residual is what the combination does beyond that.
d_idea = TAB["plan"].xs(IDEA, level="ablation").delta
CMP["d_idea"] = [d_idea[t, j] for t, j in zip(CMP.track, CMP.judge)]
CMP["residual"] = CMP.d_plan - CMP.d_abs - CMP.d_idea
print(CMP.round(1).to_string(index=False))

In [ ]:
# A cell counts as agreeing when the two deltas share a sign, or when either is
# too small for its sign to mean anything.
same = ((CMP.d_abs > 0) == (CMP.d_plan > 0)) | (CMP.d_abs.abs() < 0.5) | (CMP.d_plan.abs() < 0.5)
print(f"{len(CMP)} cells   same sign: {same.sum()}   flipped: {(~same).sum()}")
print(f"significant on abstracts: {CMP.sig_abs.sum()}   in plan format: {CMP.sig_plan.sum()}   "
      f"both: {(CMP.sig_abs & CMP.sig_plan).sum()}\n")
print(CMP[~same].round(1).to_string(index=False), "\n")
print("mean over judges:")
print(CMP.groupby(["track", "ablation"], sort=False)[["d_abs", "d_plan", "d_idea", "residual"]]
      .mean().round(1).to_string())

In [ ]:
for png in (PLAN.draw("pairwise"), PLAN.draw_absolute("pairwise")):
    print(png.relative_to(ROOT), "(+ .pdf)")
    display(Image(filename=str(png)))